## Single Run Computation: One Game of the Children's Play

Random numbers are generated to determine if it *hits* the circle, giving an estimation of $\pi$

In [29]:
import random

random.seed(0)
"""Allow reproducibility of results as the random number generator is initialized with a fixed seed rather than a random seed.
Remove to randomise results!"""

n_trials = 4000
#The more trials, the closer the estimate will be to the true value of pi.
n_hits = 0
for _ in range(n_trials):
     x, y = random.uniform(-1.0, 1.0), random.uniform (-1.0, 1.0)
     if x**2 + y**2 < 1.0:
         n_hits += 1

print(4.0 * n_hits / n_trials)

3.127


## Multirun Simulation: Many Afternoons on the Sunny Beach of Monte Carlo
We can also continuously run loops and by statistics, the mean value of $\pi$ and its standard deviation can be determined, allowing us to scale the simulation to achieve a more accurate number with the consistency of the data taken account! 

In [30]:
import random
import statistics

def estimate_pi(N):
    rand = random.random
    """ Instead of using random.uniform, we can use random.random which generates a float in the range [0.0, 1.0) and then scale it to the desired range.
    This is desirable as random.random is faster as it is a lower level function, avoiding overhead of additional calculations."""
    n_hits = 0
    for _ in range(N):
        x = rand() * 2.0 - 1.0
        y = rand() * 2.0 - 1.0
        if x**2 + y**2 < 1.0:
            n_hits += 1
    return 4.0 * n_hits / N

def main():
    n_runs = 1000
    n_trials = 400000

    # Run the estimator multiple times and collect results
    estimates = [estimate_pi(n_trials) for _ in range(n_runs)]

    # Print a summary instead of thousands of individual lines
    mean_est = statistics.mean(estimates)
    std_est = statistics.pstdev(estimates)
    print(f"Runs={n_runs} Trials={n_trials} Mean={mean_est:.6f} Std={std_est:.6f}")

if __name__ == "__main__":
    main()

Runs=1_000 Trials=400_000 Mean=3.141593 Std=0.002585


### More Efficient Scaling: NumPy
Instead of looping the function continuously, the NumPy library can be utilised to generate random numbers (*for $n_\text{hits}$*) all at once by vectorising the samples. NumPy can also run statistical analysis, hence removing the need for```py
import random
import statistics
```

In [ ]:
import numpy as np

def estimate_pi(n_trials, n_runs):
    estimates = []
    for _ in range(n_runs):
        x = np.random.uniform(-1.0, 1.0, n_trials)
        y = np.random.uniform(-1.0, 1.0, n_trials)
        n_hits = np.sum(x**2 + y**2 <= 1.0)
        estimates.append(4.0 * n_hits / n_trials)
    return np.array(estimates)
    # Convert the list of estimates to a NumPy array for efficient numerical analysis


def main():
    n_runs = 1000
    n_trials = 400000

    estimates = estimate_pi(n_trials, n_runs)

    mean_est = np.mean(estimates)
    std_est = np.std(estimates, ddof=1)
    sem = std_est / np.sqrt(n_runs)
    return mean_est, std_est, sem, n_runs, n_trials
    # Return the computed statistics for further analysis (In this case, Confidence Interval calculation)

if __name__ == "__main__":
    mean_est, std_est, sem, n_runs, n_trials = main()
    print(f"Runs = {n_runs} Trials = {n_trials} Average value = {mean_est:.6f} Standard deviation = {std_est:.6f} Standard Error = {sem:.6f}")

Runs = 1_000 Trials = 400_000 Average value = 3.141470 Standard deviation = 0.002586 Standard Error = 0.000082


### Confidence Interval: SciPy
Based on these 1000 runs, what is the uncertainty of the mean estimate? Using a 95% confidence interval, we can assume 95% of repeated confidence intervals will contain the true mean, in this case: the true value of $\pi$

In [32]:
from scipy import stats

mean_est, std_est, sem, n_runs, n_trials = main()
# Calculate the confidence interval for the estimated value of pi
confidence_level = 0.95
alpha = 1 - confidence_level
t_critical = stats.t.ppf(1 - alpha/2, df= n_runs-1)
moe = t_critical * sem
lower_bound = mean_est - moe
upper_bound = mean_est + moe

print(f"95% Confidence Interval: ({lower_bound:.6f}, {upper_bound:.6f})")

95% Confidence Interval: (3.141369, 3.141686)
